# Task 4 - Statistical Analysis

## ApexPlanet Data Analytics Internship

### Dataset
Supermarket Sales Dataset

### Objective
Perform descriptive and inferential statistical analysis to extract meaningful business insights.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats


In [ ]:
df = pd.read_csv("supermarket_sales_cleaned.csv")
df.head()

In [ ]:
print("Shape:", df.shape)
df.info()
df.columns.tolist()

In [ ]:

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:

df.isnull().sum()

In [ ]:

df.describe()

In [ ]:
for col in df.columns:
    print(col)

In [ ]:

print("Mean:", df["total"].mean())
print("Median:", df["total"].median())
print("Mode:")
print(df["total"].mode())

In [ ]:

print("Variance:", df["total"].var())
print("Standard Deviation:", df["total"].std())

In [ ]:

plt.figure(figsize=(8,5))
plt.hist(df["total"], bins=20, edgecolor="black")
plt.title("Distribution of Total Sales")
plt.xlabel("Total Sales")
plt.ylabel("Frequency")
plt.show()

In [ ]:

correlation = df.select_dtypes(include=[np.number]).corr()
correlation

In [ ]:

plt.figure(figsize=(10,6))
sns.heatmap(correlation, annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

In [ ]:

t_stat, p_value = stats.ttest_1samp(df["total"], 300)

print("T-Statistic:", t_stat)
print("P-Value:", p_value)

In [ ]:

branch_A = df[df["branch"] == "A"]["total"]
branch_B = df[df["branch"] == "B"]["total"]
branch_C = df[df["branch"] == "C"]["total"]
f_stat, p_value = stats.f_oneway(branch_A, branch_B, branch_C)
print("F-Statistic:", f_stat)
print("P-Value:", p_value)

In [ ]:

avg_sales = df.groupby("branch")["total"].mean()
print(avg_sales)

In [ ]:

avg_sales = df.groupby("branch")["total"].mean()
print(avg_sales)


# Section 2 - Time Series Analysis

### Objective
Analyze sales trends over time using the `date` column and identify patterns in total sales.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats


In [ ]:
# Convert date column to datetime

df["date"] = pd.to_datetime(df["date"])
print(df["date"].dtype)

In [ ]:
# Set date as index

df = df.set_index("date")
df.head()

In [ ]:
# Daily total sales

daily_sales = df["total"].resample("D").sum()
daily_sales.head()

In [ ]:
# Plot Daily Sales

plt.figure(figsize=(12,5))
plt.plot(daily_sales)
plt.title("Daily Total Sales")
plt.xlabel("Date")
plt.ylabel("Total Sales")
plt.show()

In [ ]:
# Monthly Total Sales

monthly_sales = df["total"].resample("ME").sum()
monthly_sales

In [ ]:
# Plot Monthly Sales

plt.figure(figsize=(8,5))
plt.plot(monthly_sales, marker="o")
plt.title("Monthly Total Sales")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.grid(True)
plt.show()

In [ ]:
# 7-Day Moving Average

moving_avg = daily_sales.rolling(window=7).mean()
moving_avg.head()

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(daily_sales, label="Daily Sales")
plt.plot(moving_avg, color="red", label="7-Day Moving Average")
plt.title("Daily Sales with 7-Day Moving Average")
plt.xlabel("Date")
plt.ylabel("Total Sales")
plt.legend()
plt.show()

In [ ]:
# Reset Index

df.reset_index(inplace=True)
df.head()

# Section 3 - Customer Segmentation

### Objective
Group customers into different segments using the K-Means clustering algorithm.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [ ]:
features = df[["unit_price", "quantity", "total"]]
features.head()

In [ ]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(features)
scaled_data[:5]

In [ ]:
# Elbow Method

wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(scaled_data)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8,5))
plt.plot(range(1,11), wcss, marker="o")
plt.title("Elbow Method")
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.show()

In [ ]:
# Apply K-Means

kmeans = KMeans(n_clusters=3, random_state=42)

df["cluster"] = kmeans.fit_predict(scaled_data)
df[["unit_price", "quantity", "total", "cluster"]].head()

In [ ]:
# Number of customers in each cluster

print(df["cluster"].value_counts())

In [ ]:
# Scatter Plot of Customer Clusters

plt.figure(figsize=(8,6))
plt.scatter(
    df["unit_price"],
    df["total"],
    c=df["cluster"],
    cmap="viridis"
)

plt.title("Customer Segments")
plt.xlabel("Unit Price")
plt.ylabel("Total Sales")
plt.show()

In [ ]:
# Average values of each cluster

cluster_summary = df.groupby("cluster")[["unit_price", "quantity", "total"]].mean()
cluster_summary

In [ ]:
print("Customer Segmentation Completed Successfully!")
print("\nBusiness Insights:")
print("- Customers were grouped into 3 clusters.")
print("- Each cluster represents a different purchasing pattern.")
print("- Businesses can use these clusters for targeted marketing.")

# Section 4 - Predictive Modeling

### Objective
Build a machine learning model to predict the total sales using Linear Regression and Decision Tree Regression.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
X = df[["unit_price", "quantity", "tax_5%", "cogs"]]
y = df["total"]
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Data:", X_train.shape)
print("Testing Data :", X_test.shape)

In [ ]:
# Linear Regression

lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

In [ ]:
# Decision Tree Regression

dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

In [ ]:
# Evaluate Models

print("Linear Regression")
print("MAE:", mean_absolute_error(y_test, lr_pred))
print("R2 Score:", r2_score(y_test, lr_pred))
print("\nDecision Tree")
print("MAE:", mean_absolute_error(y_test, dt_pred))
print("R2 Score:", r2_score(y_test, dt_pred))

In [ ]:
# Feature Importance

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": dt.feature_importances_
})

importance.sort_values(by="Importance", ascending=False)

In [ ]:
# Compare Actual vs Predicted

comparison = pd.DataFrame({
    "Actual": y_test,
    "Predicted": lr_pred
})

comparison.head(10)

In [ ]:
print("="*50)
print(" TASK 4 COMPLETED SUCCESSFULLY ")
print("="*50)
print("\nSections Completed:")
print("✔ Statistical Analysis")
print("✔ Time Series Analysis")
print("✔ Customer Segmentation")
print("✔ Predictive Modeling")
print("\nMachine Learning Models Used:")
print("- Linear Regression")
print("- Decision Tree Regressor")
print("\nProject Status: SUCCESS")